# 02 — SignalWatcher DWT + V-I demo

Visualises an appliance onset at 16 kHz. You need to run `scripts/build_signatures.py` first (this also synthesises the sample FLAC if a real UK-DALE 16 kHz file isn't present).

In [ ]:
import sys, pickle
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, soundfile as sf, matplotlib.pyplot as plt
from aerogrid.signal_watcher import SignalWatcher
from aerogrid.vi_features import SAMPLES_PER_CYCLE
from aerogrid.config import UKDALE_DIR, CACHE_DIR

flac = UKDALE_DIR / "house_1" / "mains_16khz_signatures.flac"
data, fs = sf.read(flac, always_2d=True)
voltage = data[:,0] * 300; current = data[:,1] * 15
sw = SignalWatcher.from_cache()
with (CACHE_DIR / "signatures.pkl").open("rb") as fh: pay = pickle.load(fh)
print(f"loaded {len(voltage)/fs:.1f}s of 16 kHz ({fs} Hz); signatures: {list(pay['signatures'])}")

In [ ]:
# Show raw current + DWT D1+D2 detail + envelope around the first few onsets
from datetime import datetime, timezone
onsets = sw.process_window(voltage, current, datetime(2014,1,1,tzinfo=timezone.utc))
print(f"detected {len(onsets)} onsets")
detail = sw.extract_transients(current)
env = np.sqrt(np.convolve(current**2, np.ones(SAMPLES_PER_CYCLE)/SAMPLES_PER_CYCLE, mode='same'))

t = np.arange(len(current))/fs
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(t, current, lw=0.3); axes[0].set_ylabel("I (A)")
axes[1].plot(t, detail, lw=0.3, color="C2"); axes[1].set_ylabel("DWT D1+D2 (envelope)")
axes[2].plot(t, env, color="C1"); axes[2].set_ylabel("1-cycle RMS env (A)")
for o in onsets:
    for ax in axes:
        ax.axvline((o.timestamp - datetime(2014,1,1,tzinfo=timezone.utc)).total_seconds(),
                   color={'dishwasher':'red','washing_machine':'blue'}.get(o.appliance,'k'),
                   alpha=0.4, linestyle='--')
axes[2].set_xlabel("time (s)"); plt.tight_layout(); plt.show()

In [ ]:
# V-I trajectory signatures side by side
fig, axes = plt.subplots(1, len(pay["signatures"]), figsize=(4*len(pay['signatures']), 4))
if len(pay["signatures"]) == 1: axes = [axes]
for ax, (name, sig) in zip(axes, pay["signatures"].items()):
    n = pay["n_points"]; v = sig[:n]; i = sig[n:]
    ax.plot(v, i, "o-", markersize=3); ax.set_title(f"{name} V-I trajectory")
    ax.set_xlabel("voltage (norm.)"); ax.set_ylabel("current (norm.)")
    ax.grid(alpha=0.3); ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
plt.tight_layout(); plt.show()